# SigLino-70M: Visualizing Tokens In and Tokens Out

This notebook demonstrates how the **tiiuae/siglino-70M** vision foundation model processes an image.

- **Tokens In**: The image is split into 16x16 pixel patches (visual tokens) by the image processor.
- **Tokens Out**: Each patch token is transformed into a feature vector (embedding) by the transformer encoder.

We visualize both sides to build intuition about how vision transformers tokenize and process images.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch transformers pillow matplotlib requests einops scikit-learn

In [ ]:
import torch
import torch.nn.functional as F
import requests
import types
import importlib
import matplotlib
# Use inline backend when available (Jupyter), otherwise Agg (headless)
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import einops as E
from PIL import Image
from io import BytesIO
from transformers import AutoModel, AutoImageProcessor

## 1. Load the Model and Processor

In [ ]:
model_id = "tiiuae/siglino-70M"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

print(f"Using device: {device}, dtype: {dtype}")

processor = AutoImageProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModel.from_pretrained(model_id, trust_remote_code=True).to(device, dtype=dtype)
model.eval()

# The 1-D RoPE buffer (freqs_cis) is registered as non-persistent, so it is
# not saved in the checkpoint. After from_pretrained the buffer contains
# uninitialised memory. Recompute it here.
head_dim = model.config.head_dim or model.config.dim // model.config.n_heads
d = head_dim // 2
model.register_buffer(
    "freqs_cis",
    model._precompute_freqs_cis(d, model.config),
    persistent=False,
)
print(f"Recomputed freqs_cis: NaN check = {model.freqs_cis.isnan().any().item()}")

print(f"Model loaded: {model_id}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 1b. CPU Compatibility Patch

SigLino uses `flex_attention` which requires CUDA. On CPU/MPS we monkey-patch the
forward pass to use PyTorch's `scaled_dot_product_attention` instead.
If running on CUDA, this cell is skipped.

In [ ]:
if device != "cuda":
    # Import helpers from the model's own downloaded modules
    base_package = type(model).__module__.rsplit('.', 1)[0]
    rope_mod = importlib.import_module(f"{base_package}.rope")
    attn_mod = importlib.import_module(f"{base_package}.attention")
    apply_golden_freqs = rope_mod.apply_golden_freqs_cis_to_visual_pos
    apply_3d_rope = rope_mod.apply_3d_rotary_emb
    repeat_kv = attn_mod.repeat_kv

    def cpu_forward(self, pixel_values, padding_mask=None, spatial_shapes=None,
                    output_hidden_states=False, return_dict=True, compile=True):
        N, L, _ = pixel_values.shape
        dev = pixel_values.device
        R = 1 + self.n_storage_tokens           # CLS + register tokens
        if padding_mask is None:
            padding_mask = torch.ones((N, L), dtype=pixel_values.dtype, device=dev)

        h = self.img_projector(pixel_values)
        parts = [self.cls_token.expand(N, -1, -1)]
        if self.n_storage_tokens > 0:
            parts.append(self.storage_tokens.expand(N, -1, -1))
        parts.append(h)
        h = torch.cat(parts, dim=1)
        S = h.shape[1]

        # full_mask: 1 for real tokens, 0 for padding  (N, S)
        full_mask = torch.cat(
            [torch.ones((N, R), dtype=padding_mask.dtype, device=dev), padding_mask], dim=1
        )

        # Build a 2-D boolean attention mask for SDPA  (N, 1, S, S)
        # A query at position q can attend to key at position k iff both are real.
        fm_bool = full_mask.bool()                        # (N, S)
        attn_mask_2d = fm_bool.unsqueeze(2) & fm_bool.unsqueeze(1)  # (N, S, S)
        # Convert to float mask: 0 where allowed, -inf where blocked
        attn_mask_float = torch.zeros((N, 1, S, S), dtype=h.dtype, device=dev)
        attn_mask_float.masked_fill_(~attn_mask_2d.unsqueeze(1), float("-inf"))

        # RoPE positions
        thw_pos = self._get_thw_pos(N, L, spatial_shapes, dev)
        pos_thw = E.rearrange(thw_pos, "p n s -> n s p").float()
        pm2d = torch.zeros((N, S), dtype=torch.bool, device=dev)
        pm2d[:, R:] = padding_mask.bool()
        pos_thw[:, :, 1:] = pos_thw[:, :, 1:].masked_fill(~pm2d.unsqueeze(-1), float("nan"))
        freqs_golden = apply_golden_freqs(
            self.freqs_cis_golden.to(pos_thw.dtype), pos_thw[:, :, 1:]
        )

        for layer in self.layers:
            x = h
            xn = (
                layer.attention_norm(x)
                if layer.parameterized_norm
                else F.rms_norm(x, (x.size(-1),))
            )
            att = layer.attention
            bs, sl, _ = xn.shape
            q, k, v = att.wq(xn), att.wk(xn), att.wv(xn)
            q = q.view(bs, sl, -1, att.head_dim)
            k = k.view(bs, sl, -1, att.head_dim)
            v = v.view(bs, sl, -1, att.head_dim)
            if att.use_qk_norm:
                q = F.rms_norm(q, (q.size(-1),))
                k = F.rms_norm(k, (k.size(-1),))
            k = repeat_kv(k, att.n_rep)
            v = repeat_kv(v, att.n_rep)
            q, k = apply_3d_rope(q, k, self.freqs_cis, freqs_golden, pos_thw)
            qt = q.transpose(1, 2)   # (B, H, S, D)
            kt = k.transpose(1, 2)
            vt = v.transpose(1, 2)

            # SDPA with proper attention mask
            out = F.scaled_dot_product_attention(qt, kt, vt, attn_mask=attn_mask_float)

            # Sink attention: compute log-sum-exp manually with the same mask
            scale = qt.shape[-1] ** -0.5
            scores = torch.matmul(qt, kt.transpose(-2, -1)) * scale  # (B, H, S, S)
            scores = scores + attn_mask_float          # mask out invalid pairs
            lse = torch.logsumexp(scores, dim=-1)      # (B, H, S)
            sinks_bhl = att.sinks.view(1, -1, 1)
            sink_scale = torch.sigmoid(lse - sinks_bhl)
            out = (out * sink_scale.unsqueeze(-1)).to(out.dtype)

            out = E.rearrange(out, "b h s d -> b s (h d)").contiguous()
            h2 = x + att.wo(out)
            hn = (
                layer.ffn_norm(h2)
                if layer.parameterized_norm
                else F.rms_norm(h2, (h2.size(-1),))
            )
            h = h2 + (layer.moe(hn) if layer.moe_enabled else layer.feed_forward(hn))

        h = self.norm(h)
        cls_f, patch_f = h[:, 0], h[:, R:]
        dp = self.dinov3_adapter(patch_f)
        sp = self.siglip2_adapter(patch_f)
        dc = self.dinov3_adapter(cls_f)
        h_sig = self.siglip2_adapter(h)
        sc = self.siglip2_multihead_attention_pooling_head(h_sig, full_mask.reshape(-1))
        return {
            "last_hidden_state": h,
            "patch_features": {"dinov3": dp, "siglip2": sp, "siglino": patch_f},
            "summary_features": {"dinov3": dc, "siglip2": sc, "siglino": cls_f},
            "hidden_states": None,
        }

    model.forward = types.MethodType(cpu_forward, model)
    print("Applied CPU-compatible forward pass (SDPA instead of flex_attention).")
else:
    print("CUDA available — using native flex_attention.")

## 2. Load a Sample Image

In [ ]:
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

if response.status_code == 200 and "image" in response.headers.get("Content-Type", ""):
    image = Image.open(BytesIO(response.content)).convert("RGB")
else:
    print("Could not fetch image from URL — generating a synthetic test image.")
    np.random.seed(42)
    image = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))

# Crop to square so the patch grid has no padding tokens (cleaner demo)
w, h = image.size
side = min(w, h)
left, top = (w - side) // 2, (h - side) // 2
image = image.crop((left, top, left + side, top + side))

print(f"Image size (center-cropped to square): {image.size[0]}x{image.size[1]} pixels")

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title("Original Image")
plt.axis("off")
plt.show()

## 3. Tokens In — Preprocessing and Patch Tokenization

The SigLino image processor resizes the image and splits it into 16x16 pixel patches.
Each patch is flattened into a vector of `3 * 16 * 16 = 768` values.
The output `pixel_values` tensor has shape `[batch, num_patches, 768]` — already tokenized!

In [ ]:
inputs = processor(image, return_tensors="pt").to(device)

pixel_values = inputs["pixel_values"].to(dtype)
spatial_shapes = inputs["spatial_shapes"]  # e.g. [[16, 16]] = grid of patches

batch_size, total_tokens, patch_dim = pixel_values.shape
grid_h, grid_w = spatial_shapes[0].tolist()
num_patches = grid_h * grid_w  # real (non-padding) tokens
patch_size = 16  # SigLino uses 16x16 patches
img_h, img_w = grid_h * patch_size, grid_w * patch_size

print("=" * 55)
print("TOKENS IN")
print("=" * 55)
print(f"pixel_values shape:  {tuple(pixel_values.shape)}")
print(f"  - batch size:      {batch_size}")
print(f"  - total tokens:    {total_tokens}  (incl. {total_tokens - num_patches} padding)")
print(f"  - real patches:    {num_patches}  ({grid_h} x {grid_w} grid)")
print(f"  - patch dim:       {patch_dim}  (3 channels x {patch_size} x {patch_size} pixels)")
print(f"")
print(f"Each input token is a flattened {patch_size}x{patch_size} pixel patch.")
print(f"The full image ({img_w}x{img_h}) is represented as {num_patches} tokens.")

In [ ]:
# Reconstruct the image from the tokenized patches to visualize the grid
# Use only the real (non-padding) patches
patches_np = pixel_values[0, :num_patches].detach().cpu().float().numpy()  # (num_patches, 768)

# Reshape: (num_patches, 768) -> (grid_h, grid_w, 3, 16, 16) -> (H, W, 3)
patches_grid = patches_np.reshape(grid_h, grid_w, 3, patch_size, patch_size)
img_reconstructed = np.transpose(patches_grid, (0, 3, 1, 4, 2)).reshape(img_h, img_w, 3)

# Normalize to [0, 1] for display
img_display = (img_reconstructed - img_reconstructed.min()) / (img_reconstructed.max() - img_reconstructed.min())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: processed image with patch grid overlay
axes[0].imshow(img_display)
axes[0].set_title(f"Tokens In: {grid_w}x{grid_h} = {num_patches} patches")
for i in range(grid_h + 1):
    axes[0].axhline(y=i * patch_size, color='red', linewidth=0.5, alpha=0.7)
for j in range(grid_w + 1):
    axes[0].axvline(x=j * patch_size, color='red', linewidth=0.5, alpha=0.7)
axes[0].axis("off")

# Right: show a few individual patches
sample_positions = [
    (grid_h // 4, grid_w // 4),
    (grid_h // 2, grid_w // 2),
    (3 * grid_h // 4, 3 * grid_w // 4),
]
colors = ['cyan', 'yellow', 'lime']
for idx, (row, col) in enumerate(sample_positions):
    patch = img_display[row*patch_size:(row+1)*patch_size, col*patch_size:(col+1)*patch_size]
    ax_inset = fig.add_axes([0.55 + idx * 0.15, 0.55, 0.12, 0.3])
    ax_inset.imshow(patch)
    ax_inset.set_title(f"Token ({row},{col})", fontsize=9)
    ax_inset.axis("off")
    rect = mpatches.Rectangle(
        (col * patch_size, row * patch_size), patch_size, patch_size,
        linewidth=2, edgecolor=colors[idx], facecolor='none'
    )
    axes[0].add_patch(rect)

axes[1].axis("off")
axes[1].set_title("Sample input tokens (patches)", fontsize=11)
plt.tight_layout()
plt.show()

## 4. Tokens Out — Model Output Embeddings

We pass the tokenized patches through the model. SigLino produces:
- **Patch features**: one embedding vector per input token
- **Summary features**: a single global embedding for the whole image

Available feature branches:
- `siglino` (512d) — fused student representation
- `siglip2` (1152d) — SigLIP2 teacher branch
- `dinov3` (1024d) — DINOv3 teacher branch

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

print("=" * 55)
print("TOKENS OUT")
print("=" * 55)

print("\nPatch Features (per-token output):")
for name, feats in outputs["patch_features"].items():
    # Slice to real patches only (exclude padding)
    real = feats[:, :num_patches]
    print(f"  {name:>10}: {str(tuple(real.shape)):>20}  "
          f"= {real.shape[1]} tokens x {real.shape[2]}d")

print("\nSummary Features (global image embedding):")
for name, feats in outputs["summary_features"].items():
    shape = tuple(feats.shape)
    dim = shape[-1]
    print(f"  {name:>10}: {str(shape):>20}  = {dim}d vector")

sig_out_dim = outputs["patch_features"]["siglino"].shape[2]
print("\n" + "=" * 55)
print("TOKENS IN vs TOKENS OUT")
print("=" * 55)
print(f"  IN:  {num_patches} tokens, each {patch_dim}d  (raw pixel patches)")
print(f"  OUT: {num_patches} tokens, each {sig_out_dim}d  (learned embeddings)")
print(f"")
print(f"  The transformer maps each {patch_dim}-dim raw patch")
print(f"  to a {sig_out_dim}-dim semantic feature vector.")

## 5. Visualize Output Token Embeddings

We reshape the output embeddings back to the spatial grid and use PCA to reduce them to 3 channels (RGB). This lets us see what the model has learned about different image regions.

In [ ]:
from sklearn.decomposition import PCA

feature_types = ["siglino", "siglip2", "dinov3"]
fig, axes = plt.subplots(1, len(feature_types) + 1, figsize=(20, 5))

axes[0].imshow(img_display)
axes[0].set_title("Original Image")
axes[0].axis("off")

for idx, feat_name in enumerate(feature_types):
    patch_feats = outputs["patch_features"][feat_name][0, :num_patches].detach().cpu().float().numpy()
    n_tok, embed_dim = patch_feats.shape

    pca = PCA(n_components=3)
    pca_result = pca.fit_transform(patch_feats)
    for c in range(3):
        ch = pca_result[:, c]
        pca_result[:, c] = (ch - ch.min()) / (ch.max() - ch.min() + 1e-8)

    pca_image = pca_result.reshape(grid_h, grid_w, 3)
    axes[idx + 1].imshow(pca_image)
    axes[idx + 1].set_title(f"Tokens Out: {feat_name}\n({n_tok} tokens x {embed_dim}d)")
    axes[idx + 1].axis("off")

plt.suptitle("Input Image vs Output Token Embeddings (PCA -> RGB)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Token-level Similarity Map

Pick a reference output token and compute cosine similarity against all other output tokens. This shows which image regions the model considers semantically similar.

In [ ]:
patch_feats = outputs["patch_features"]["siglino"][0, :num_patches].detach().cpu().float()  # (num_patches, 512)

ref_row, ref_col = grid_h // 2, grid_w // 2
ref_idx = ref_row * grid_w + ref_col
ref_token = patch_feats[ref_idx].unsqueeze(0)

similarities = F.cosine_similarity(ref_token, patch_feats, dim=1)
sim_map = similarities.numpy().reshape(grid_h, grid_w)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(img_display)
rect = mpatches.Rectangle(
    (ref_col * patch_size, ref_row * patch_size), patch_size, patch_size,
    linewidth=3, edgecolor='red', facecolor='none'
)
axes[0].add_patch(rect)
axes[0].set_title(f"Reference token at ({ref_row}, {ref_col})")
axes[0].axis("off")

im = axes[1].imshow(sim_map, cmap='hot', interpolation='nearest')
axes[1].set_title("Cosine Similarity Map\n(output token space)")
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046)

axes[2].imshow(img_display)
axes[2].imshow(
    np.kron(sim_map, np.ones((patch_size, patch_size))),
    cmap='hot', alpha=0.5
)
axes[2].set_title("Similarity Overlay on Image")
axes[2].axis("off")

plt.suptitle("Token-level Cosine Similarity (siglino features)", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Summary

| | Tokens In | Tokens Out |
|---|---|---|
| **What** | Image patches (16x16 pixels) | Feature embeddings per patch |
| **Count** | H/16 x W/16 patches | Same number of tokens |
| **Dimensionality** | 3 x 16 x 16 = 768 raw pixel values | 512d (siglino), 1152d (siglip2), 1024d (dinov3) |
| **Meaning** | Raw pixel colors | Learned semantic features |
| **Additional outputs** | -- | Summary features (1 global vector per image) |